# Principal Components and Factor Models

Companion notebook for the [slides](https://busi521.kerryback.com/latex/PCA_Factors.pdf).  We fit 3-factor models to the 25 Fama-French size/book-to-market sorted portfolios using factor analysis and principal components, construct factor-mimicking portfolios, and compare maximum Sharpe ratios.

## Data

We use monthly excess returns on the 25 size and book-to-market sorted portfolios from the Fama-French data library.

In [ ]:
import pandas as pd
import numpy as np
import pandas_datareader as pdr
import statsmodels.api as sm

startdate = '1980-01-01'

# 25 size/BM sorted portfolio returns
Rets = pdr.DataReader('25_Portfolios_5x5', 'famafrench', start=startdate)[0]

# Fama-French factors and risk-free rate
FF = pdr.DataReader('F-F_Research_Data_Factors', 'famafrench', start=startdate)[0]

# convert to excess returns
Rets = Rets.apply(lambda x: x - FF['RF'])

## Factor Analysis

See slides: *Factor Analysis (ML Estimation)* and *Identification*.

Maximum likelihood estimation recovers the loading matrix $\hat{B}$ and diagonal residual covariance $\hat{D}$.  The factors are then inferred by GLS:

$$f_t = (\hat{B}'\hat{D}^{-1}\hat{B})^{-1}\hat{B}'\hat{D}^{-1}(R_t - \hat{\mu})$$

In [ ]:
from sklearn.decomposition import FactorAnalysis

Factors_FA = FactorAnalysis(n_components=3).fit_transform(Rets)
Factors_FA = pd.DataFrame(Factors_FA, index=Rets.index, columns=range(3))

print('Factor means (should be ~0):')
print(Factors_FA.mean())
print('\nFactor covariance (should be ~identity):')
print(Factors_FA.cov())

### Factor-Mimicking Portfolios (Factor Analysis)

See slides: *Constructing FMPs: Factor Analysis*.

The factors are spanned by the returns (they are computed by GLS regression on the returns).  To construct factor-mimicking portfolios, we regress each factor on the returns and normalize the coefficients to sum to one.

In [ ]:
FMP_FA = pd.DataFrame(dtype=float, index=Factors_FA.index, columns=Factors_FA.columns)
X = sm.add_constant(Rets)
for i in range(3):
    wts = sm.OLS(Factors_FA[i], X).fit().params[1:]
    wts = wts / wts.sum()
    FMP_FA[i] = Rets @ wts

FMP_FA.head()

## Principal Components

See slides: *Eigenvectors of the Covariance Matrix* and *PCA as a Factor Model*.

PCA uses the eigenvectors of $\Sigma = \text{Cov}(R_t)$.  Let $C_1$ contain the $K$ eigenvectors with the largest eigenvalues.  The factors are

$$f_t = (R_t - \hat{\mu})\,C_1$$

PCA does not force residuals to be uncorrelated.  Instead, it makes residuals small by dropping eigenvectors with small eigenvalues.

In [ ]:
from sklearn.decomposition import PCA

Factors_PCA = PCA(n_components=3).fit_transform(Rets)
Factors_PCA = pd.DataFrame(Factors_PCA, index=Rets.index, columns=range(3))
Factors_PCA.head()

We can equivalently compute the eigenvectors directly.  Note that `linalg.eigh` sorts from smallest to largest eigenvalue, so we reverse.

In [ ]:
E, C = np.linalg.eigh(Rets.cov())
C = pd.DataFrame(C, index=Rets.columns, columns=range(25))
C1 = C[[24, 23, 22]]
C1.columns = [0, 1, 2]

### Variance Explained

See slide: *How Much Variance Is Explained?*

Each eigenvalue equals the variance of the corresponding principal component.  The fraction of total variance explained by each component:

In [ ]:
fractions = E / E.sum()
print('Fraction of variance explained by each component (largest to smallest):')
print(fractions[::-1][:5])
print(f'\nFirst 3 components explain {fractions[-3:].sum():.1%} of total variance')

### Factor-Mimicking Portfolios (PCA)

See slide: *Constructing FMPs: PCA*.

The eigenvector entries are the portfolio weights.  Normalizing so the weights sum to one:

In [ ]:
FMP_PCA = Rets @ (C1 / C1.sum())
FMP_PCA.head()

## Comparing Maximum Sharpe Ratios

See slide: *Maximum Sharpe Ratios*.

The maximum Sharpe ratio from $K$ excess-return factors with mean $\mu_f$ and covariance $\Sigma_f$ is

$$\text{SR}^* = \sqrt{\mu_f'\Sigma_f^{-1}\mu_f}$$

In [ ]:
def max_sharpe(df):
    mu = df.mean()
    Sigma = df.cov()
    return np.sqrt(mu.T @ np.linalg.inv(Sigma) @ mu)

print(f'Fama-French SR:  {max_sharpe(FF[["Mkt-RF", "SMB", "HML"]]):.4f}')
print(f'FA mimicking SR: {max_sharpe(FMP_FA):.4f}')
print(f'PCA mimicking SR: {max_sharpe(FMP_PCA):.4f}')